In [14]:
from datasets import load_dataset
import matplotlib.pyplot as plt
import pandas as pd
import random

random.seed(42)

ds = load_dataset("bridge-benchmark-2026/BRIDGE")
df  = ds["train"].to_pandas()

print("Rows and columns:", df.shape)
print("\nColumns:", df.columns.tolist())

df.info()

Rows and columns: (19421, 7)

Columns: ['sentence_id', 'text', 'bias_label', 'source', 'register', 'era', 'domain']
<class 'pandas.DataFrame'>
RangeIndex: 19421 entries, 0 to 19420
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   sentence_id  19421 non-null  int64
 1   text         19421 non-null  str  
 2   bias_label   19421 non-null  str  
 3   source       19421 non-null  str  
 4   register     19421 non-null  str  
 5   era          19421 non-null  str  
 6   domain       19421 non-null  str  
dtypes: int64(1), str(6)
memory usage: 7.0 MB


In [15]:

display(df.drop(columns="text").head())

label_counts = df["bias_label"].value_counts(dropna=False)
label_summary = pd.DataFrame({
    "count": label_counts,
    "percent": (label_counts / len(df) * 100).round(2)
})

display(label_summary)



,sentence_id,bias_label,source,register,era,domain
0,0,harmful,Social Bias Frames,informal,contemporary,racial/cultural
1,1,harmful,Social Bias Frames,informal,contemporary,disability
2,2,harmful,Social Bias Frames,informal,contemporary,racial/cultural
3,3,harmful,Social Bias Frames,informal,contemporary,racial/cultural
4,4,harmful,Social Bias Frames,informal,contemporary,racial/cultural


,count,percent
bias_label,,
harmful,7799,40.16
harmless,6400,32.95
antibias,5222,26.89


In [16]:
inspect = df.copy()

# find missing or blank text
text = inspect["text"].astype("string")
blank_text = text.fillna("").str.strip().eq("")

# normalize text
inspect["text_key"] = (
    text.fillna("")
    .str.normalize("NFKC")
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
)

# keep valid rows
valid_text = inspect.loc[~blank_text].copy()

# Count the distinct labels assigned to each normalized text.
labels_per_text = (
    valid_text.groupby("text_key")["bias_label"].nunique()
)

# Identify texts that have more than one label.
conflicting_keys = labels_per_text[
    labels_per_text > 1
].index

# Collect every row belonging to those conflicting groups.
conflicts = valid_text[
    valid_text["text_key"].isin(conflicting_keys)
].copy()

print("Conflicting groups:", conflicts["text_key"].nunique())
print("Affected rows:", len(conflicts))

Conflicting groups: 51
Affected rows: 163
